# Project pipeline — Northern Bahr el Ghazal flood forecasting (JBG060 / ZHL)



Run the notebook top-to-bottom from the **repository root**. Model selection between the grid cells is done
manually by diffing the `cv_metrics_summary_aweil.csv` tables in the respective
`LightGBM_v1/outputs/outputs_*/tables/` folders.

## 1. Pre-processing

The model consumes:

| Input | Producer | Used by |
|---|---|---|
| `EDA_flood_masks/outputs/tables/weekly_county_floods.csv` | **flood-label cell (this section)** | flood labels (`data_loader.load_flood_labels`) |
| `EDA_hydrometeorology/outputs_county/tables/aweil_monthly_agricultural_exposure.csv` | **exposure cell (this section)** | exposure context |
| `raw_data/Administrative boundaries/ssd_admin2.geojson` | raw data | county boundaries |
| `raw_data/rainfall and runoff`, `raw_data/Darthmouth Flood Observatory`, `raw_data/Water levels lakes` | raw data | ERA5 precipitation/runoff, river gauge, Lake Albert level |
| `processing_data/evapotranspiration/ET_<year>_9.475N_30.725E_processed.csv` | **`process_et_NBeG.py` (this section)** | ET0 feature block |

Three pre-processing cells below turn the raw products into the two derived
tables and the ET0 feature block:

1. **ET0 cell** — `process_et_NBeG.py` processes the raw AgERA5 ET NetCDFs into
   the annual CSVs.
2. **Flood-label cell** — re-aggregates the raw flood-mask Parquets into the
   weekly county table with the exact `EDA_flood_masks` EDA chain.
3. **Exposure cell** — samples the 2024 flood detections against the ASAP crop
   and rangeland masks with the exact `EDA_hydrometeorology` EDA chain.

Any missing real source raises a
`RuntimeError` / `FileNotFoundError`. The preflight below checks the raw inputs
up front, and every later run prints the full provenance report (one row per
real source) to the console.

In [ ]:
import sys

%pip install -r requirements.txt

  Using cached tabpfn-9.0.0-py3-none-any.whl.metadata (22 kB)
  Using cached torch-2.14.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached skrub-0.10.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached hf_xet-1.7.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.50.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.5 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.me

In [1]:
from pathlib import Path

from LightGBM_v1 import config

# External raw inputs: a missing real source raises FileNotFoundError or
# RuntimeError inside the pipeline, so check them here before pre-processing.
raw_inputs = {
    "admin-2 boundaries": config.BOUNDARY_ADMIN2,
    "raw flood-mask Parquets": config.RAW_DATA / "flood_masks",
    "ASAP crop and rangeland masks": config.RAW_DATA / "farmland",
    "raw ERA5 rainfall/runoff": config.RAW_DATA / "rainfall and runoff",
    "Dartmouth gauge": config.GAUGE_ROOT,
    "Lake Albert levels": config.LAKE_ROOT,
    "raw AgERA5 ET NetCDFs": config.RAW_DATA / "evapotranspiration",
}
print("External raw inputs:")
for label, path in raw_inputs.items():
    print(f"  [{'OK' if Path(path).exists() else 'MISSING':>7}] {label}: {path}")

# The pre-processing cells below regenerate these derived artifacts.
generated = {
    "weekly flood labels (flood-label cell)": config.AWEIL_WEEKLY_FLOODS_CSV,
    "monthly agricultural exposure (exposure cell)": config.AWEIL_MONTHLY_EXPOSURE_CSV,
}
print("\nRegenerated by the pre-processing cells below:")
for label, path in generated.items():
    print(f"  [{'OK' if Path(path).exists() else 'MISSING':>7}] {label}: {path}")

n_et = len(list(Path(config.ET0_ROOT).glob("ET_*_processed.csv")))
missing_et = len(config.YEARS) - n_et
et_state = "OK" if missing_et == 0 else "MISSING"
print(f"  [{et_state:>7}] processed ET0 CSVs: {n_et}/{len(config.YEARS)} years in {config.ET0_ROOT}")

todo = [label for label, path in generated.items() if not Path(path).exists()]
if missing_et:
    todo.append(f"{missing_et} ET0 year(s)")
if todo:
    print("-> run the pre-processing cells below for: " + "; ".join(todo))

External raw inputs:
  [     OK] admin-2 boundaries: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/Administrative boundaries/ssd_admin2.geojson
  [     OK] raw flood-mask Parquets: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/flood_masks
  [     OK] ASAP crop and rangeland masks: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/farmland
  [     OK] raw ERA5 rainfall/runoff: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/rainfall and runoff
  [     OK] Dartmouth gauge: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/Darthmouth Flood Observatory
  [     OK] Lake Albert levels: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/Water levels lakes
  [     OK] raw AgERA5 ET NetCDFs: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/evapotranspiration

Regenerated by the pre-processing cells below:
  [     OK] weekly flood labels (flood-label cell): /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/EDA_flood_masks/outputs/t

In [ ]:
# Process the raw AgERA5 ET NetCDFs into the annual CSVs.
!{sys.executable} process_et_NBeG.py

In [ ]:
# Regenerate the weekly flood-label table.
# Missing raw files raise FileNotFoundError; nothing is substituted.
from EDA_flood_masks.flood_eda import (
    load_flood_observations,
    load_study_counties,
    summarise_weekly,
    validate_outputs,
)

flood_root = config.RAW_DATA / "flood_masks"
label_counties = load_study_counties(config.BOUNDARY_ADMIN2)
label_counties = label_counties.sort_values("county").reset_index(drop=True)
label_observations = load_flood_observations(flood_root, label_counties)
weekly_labels = summarise_weekly(label_observations)
validate_outputs(label_observations, weekly_labels)

config.AWEIL_WEEKLY_FLOODS_CSV.parent.mkdir(parents=True, exist_ok=True)
weekly_labels.to_csv(config.AWEIL_WEEKLY_FLOODS_CSV, index=False)
print(f"{len(weekly_labels):,} county-weeks with detections -> {config.AWEIL_WEEKLY_FLOODS_CSV}")
print(
    f"Detections span {label_observations['date'].min():%Y-%m-%d} "
    f"to {label_observations['date'].max():%Y-%m-%d}"
)

In [ ]:
# Regenerate the monthly agricultural exposure table.
# Missing raw files raise an error; nothing is substituted.
from pathlib import Path

import numpy as np

from EDA_hydrometeorology.hydrological_analysis import (
    agricultural_baseline,
    load_aweil_counties,
    load_aweil_flood_observations,
    seasonal_agricultural_exposure,
)

flood_root = config.RAW_DATA / "flood_masks"
mask_paths = {
    "crop": str(config.RAW_DATA / "farmland" / "asap_mask_crop_v04.tif"),
    "rangeland": str(config.RAW_DATA / "farmland" / "asap_mask_rangeland_v04.tif"),
}
for land_type, path in mask_paths.items():
    if not Path(path).is_file():
        raise RuntimeError(f"Missing {land_type} exposure raster: {path}")

exposure_counties = load_aweil_counties(config.BOUNDARY_ADMIN2)
exposure_observations = load_aweil_flood_observations(
    flood_root, exposure_counties, years=np.array([2024])
)
exposure_baseline = agricultural_baseline(exposure_counties, mask_paths)
exposure = seasonal_agricultural_exposure(exposure_observations, exposure_counties, mask_paths)
exposure = exposure.merge(exposure_baseline, on="county", how="left")
exposure["crop_exposed_percent"] = 100 * exposure["crop_exposed_hectares"] / exposure["crop_hectares"]
exposure["rangeland_exposed_percent"] = 100 * exposure["rangeland_exposed_hectares"] / exposure["rangeland_hectares"]

config.AWEIL_MONTHLY_EXPOSURE_CSV.parent.mkdir(parents=True, exist_ok=True)
exposure.to_csv(config.AWEIL_MONTHLY_EXPOSURE_CSV, index=False)
print(f"{len(exposure)} county x month x flood-type rows -> {config.AWEIL_MONTHLY_EXPOSURE_CSV}")

## 2. Level 1 — TabPFN zero-shot reference


pretrained TabPFN classifiers/regressors that never train on our data, giving an
honest, tuning-free judge of how much signal the feature matrix carries.

- Outputs → `TabPFN_v1/outputs/outputs_v3/tables/`.
- Hyperparameters are not tunable; `--params` has no effect here.

In [ ]:
from LightGBM_v1.run import main

# Zero-shot TabPFN reference (outputs -> TabPFN_v1/outputs/outputs_v3/tables/).
main("tabpfn")

## 3. Level 1 — baseline features, baseline hyperparameters

Train lightGBM with the baseline feature set and baseline hyperparameters

- Early stopping on each fold's validation year; boundary week purged per fold.
- The climatology and Markov-persistence references are re-estimated **inside
  each fold's training window** — they never see validation/test weeks.
- Outputs → `LightGBM_v1/outputs/outputs_baseline/tables/`.

In [ ]:
from LightGBM_v1.run import main


# (outputs -> LightGBM_v1/outputs/outputs_baseline/tables/).
main("lightgbm_baseline")

## 4. Level 1 — Optuna tuning on the baseline features

`objective="all_heads"` runs three **resumable** single-head studies that share
one feature matrix (trial #0 of each re-enqueues today's defaults as an anchor):

| Stage | Study | Head | Objective |
|---|---|---|---|
| 1 | `lgbm_aweil_baseline_v3_brier` | det | Brier |
| 2 | `lgbm_aweil_baseline_v3_pinball` | quantiles | mean pinball loss |
| 3 | `lgbm_aweil_baseline_v3_dur` | duration | per-horizon Brier |

50 trials per stage (timeout 3600 s per stage); studies live in the SQLite store
`LightGBM_v1/outputs/tuning/studies.db`, so re-running this cell resumes the same
study and adds trials. The per-head winners are merged into
`outputs/tuning/best_params_lgbm_aweil_baseline_v3_all_heads.json`, and
`evaluate=True` additionally refits the merged winners and scores the frozen test
years into `cv_metrics_tuned_lgbm_aweil_baseline_v3_all_heads.csv` (the tuned
duration metrics come from the full pipeline run in the next section).

**Hand-off to section 5:** `run.py` does not read the JSON at run time —
`params_name="baseline_optuna"` uses the winners hard-coded in
`LightGBM_v1/lgbm_model.py::PARAM_SETS`. After (re-)tuning, copy the new
`det` / `quantiles` / `duration` values from the JSON into
`PARAM_SETS["baseline_optuna"]` before running section 5.

In [ ]:
from LightGBM_v1.tuning import tune_lightgbm

study_baseline = tune_lightgbm(
    feature_set="baseline",
    objective="all_heads",  # det (Brier) + quantiles (pinball) + duration (Brier)
    n_trials=50,            # per stage
    evaluate=True,          # also refit the merged winners on the frozen test years
)

## 5. Level 1 — baseline features with the Optuna winners

Re-runs the identical frozen protocol with the tuned per-head parameters. Outputs
go to their own folder, `LightGBM_v1/outputs/outputs_baseline_optuna/tables/`, so
the frozen baseline outputs stay untouched and comparable. If you tuned with new
parameters in section 4, first copy the JSON winners into `PARAM_SETS` as
described above; as-is, this cell reproduces the completed tuned run.

In [ ]:
from LightGBM_v1.run import main

# Tuned winners from PARAM_SETS["baseline_optuna"]
# (outputs -> LightGBM_v1/outputs/outputs_baseline_optuna/tables/).
main("lightgbm_baseline", "baseline_optuna")

## 6. Level 1 — engineered features, v3 defaults

The engineered grid cell (`features_fe.py`): starts from the frozen v3 matrix and
adds `cnty_tp_w7` / `cnty_ro_w7` (7-day ERA5 precipitation/runoff rolling sums
over each county's **own** bounding box) and the `county` identity as a
categorical feature, and drops `year` (the cyclic `month` / `week_of_year`
encodings stay) → 52 columns. Same folds, embargo and frozen v3 hyperparameters
(`PARAM_SETS["fe_baseline"]` = no overrides); outputs →
`LightGBM_v1/outputs/outputs_fe/tables/`.

In [ ]:
from LightGBM_v1.run import main

# Engineered feature set with the frozen v3 defaults
# (outputs -> LightGBM_v1/outputs/outputs_fe/tables/).
main("lightgbm_fe", "fe_baseline")

## 7. Level 1 — Optuna tuning on the engineered features

Same three-stage all-heads design as section 4, on the 52-column engineered
matrix: studies `lgbm_aweil_fe_v3_{brier,pinball,dur}` in
`outputs/tuning/studies.db`, merged winners in
`outputs/tuning/best_params_lgbm_aweil_fe_v3_all_heads.json`, and (with
`evaluate=True`) the tuned test-year metrics in
`cv_metrics_tuned_lgbm_aweil_fe_v3_all_heads.csv`.

**Hand-off to section 8:** after (re-)tuning, copy the JSON's
`det` / `quantiles` / `duration` groups into `PARAM_SETS["fe_optuna"]` in
`LightGBM_v1/lgbm_model.py`.

In [ ]:
from LightGBM_v1.tuning import tune_lightgbm

study_fe = tune_lightgbm(
    feature_set="fe",
    objective="all_heads",  # det (Brier) + quantiles (pinball) + duration (Brier)
    n_trials=50,            # per stage
    evaluate=True,          # also refit the merged winners on the frozen test years
)

## 8. Level 1 — engineered features with the Optuna winners

Re-runs the identical frozen protocol with the tuned parameters on the engineered
matrix; outputs → `LightGBM_v1/outputs/outputs_fe_optuna/tables/`.

In [ ]:
from LightGBM_v1.run import main

# Tuned winners from PARAM_SETS["fe_optuna"]
# (outputs -> LightGBM_v1/outputs/outputs_fe_optuna/tables/).
main("lightgbm_fe", "fe_optuna")

## 9. Level 1 — final fit mode

 `LightGBM_v1.run.final_fit("lightgbm_fe", "fe_baseline")` retrains
the chosen configuration once — the CV protocol's expanding-window logic taken one
fold further: train on everything before the final year (2000–2024), early-stopping
on the final year (2025). It then forecasts the **latest feature-complete week** —
with inputs ending Dec 2025 that is **2025-12-29** — reporting `det_prob` plus
`q10/q50/q90` for that week and `det_prob_h1/h2` from the duration heads (the honest
forecast of the following two weeks; h3–h4 lose to climatology and persistence in
`duration_metrics_aweil.csv`).

Later (2026) weeks are deliberately **not** forecast: they would need weather data
beyond Dec 2025, and partial rolling windows would silently change what the features
mean — the completeness gate in `run.py` drops such rows instead. Once 2026 ERA5/ET0
and flood labels are added, this same cell forecasts the then-current week.

Output: `LightGBM_v1/outputs/outputs_final_fit/tables/final_forecast_aweil.csv`
(columns `county, week, det_prob, q10, q50, q90, det_prob_h1, det_prob_h2`), which
feeds the section-11 maps via `climatology_maps.forecast_maps --predictions PATH`.


In [ ]:
from LightGBM_v1.run import final_fit

# Operational product: retrain lightgbm_fe (frozen v3 defaults) on 2000-2024
# with early stopping on 2025, then forecast the latest feature-complete week
# (2025-12-29) plus duration horizons h1-h2
# (output -> LightGBM_v1/outputs/outputs_final_fit/tables/final_forecast_aweil.csv).
final_fit("lightgbm_fe", "fe_baseline")


## 10. Level 2 — training-validation (pixel scale)

What exists today for Task B is the **pixel-climatology spatial prior** and its
honest evaluation; a trained per-cell LightGBM or U-Net is plan-only
(`climatology_maps/UNET_PLAN.md`).

1. `pixel_climatology.py` — per ~232 m grid cell and calendar month, from the raw
   flood-mask Parquets (`raw_data/flood_masks/compact_*/flood_events_h20v08_*.parquet`,
   2000–2025): flood frequency, frequency **given the county flooded** (the spatial
   signature used to spread a county flood across its cells), and the county-month
   probability `(k+0.5)/(n+1)`. Cached in `climatology_maps/outputs/climatology_data/`.
2. `evaluate_pixel_maps.py` — the honest validation: scores the LightGBM ×
   pixel-climatology maps on the OOF test weeks only (per-fold climatology trained
   on 2000–2008 / 2000–2014 / 2000–2020; 2025 excluded) with pixel-week Brier/BSS
   vs climatology and persistence, AUROC/PR-AUC, the threshold family
   (CSI/FAR/POD/…), weekly flooded-area MAE/bias and reliability — written to
   `climatology_maps/outputs/forecast_maps/PIXEL_PERFORMANCE.md`.

Predictions default to the first committed LightGBM table found among `outputs_fe`
→ `outputs_fe_optuna` → `outputs_baseline_optuna` → `outputs_baseline` →
`outputs_v3` (`cv_test_predictions_aweil.csv`); pass `--predictions <path>` to
score a specific run's table instead.

In [ ]:
import sys

# One-time build of the per-pixel climatology cache (~30 s; needs the raw
# flood-mask Parquets). Add --rebuild to force a fresh build.
!{sys.executable} -m climatology_maps.pixel_climatology

In [ ]:
import sys

# Honest pixel-level validation of the LightGBM x pixel-climatology maps on the
# OOF test weeks (per-fold trained climatology; 2025 excluded).
!{sys.executable} -m climatology_maps.evaluate_pixel_maps

## 11. Level 2 — final fit mode

**Not implemented yet.** There is no trained Level-2 model to put into a final-fit
mode (the per-cell LightGBM and the U-Net are plan-only). The Level-1 final-fit
forecast now exists (section 9: `outputs_final_fit/tables/final_forecast_aweil.csv`)
and is what this cell would consume; the missing piece is the per-county → pixel
step itself. Once implemented, this cell would produce the weekly pixel map for the
forecast week and hand it to the risk step below.


In [ ]:
# TODO(level 2 final fit): no trained per-cell/U-Net model exists yet — leave
# empty until the Level-2 model is implemented (the Level-1 final-fit forecast
# is ready — section 9).


## 12. Calculate risks

**Not implemented yet** (deliberately left for later). The rule-based fusion from
`model_plan.md`: Task A decides *whether/how much* (county `det_prob`, `q50`),
Task B only decides *where* (normalised cell probabilities spread Task A's
predicted area across the county's cells), the result is multiplied by the
cattle-density exposure map, and the predicted county area is compared against
county-month climatology thresholds for the risk tier (above the monthly mean →
tier 1, above p80 → tier 2, above p95 → tier 3). Products: `advisories.csv` +
`movement_advice.csv`.

In [ ]:
# TODO(risks): fuse Level 1 county forecasts with the Level 2 spatial pattern and
# the cattle-density map — no existing function implements this yet.